# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
%pip -q install duckdb

import os, getpass
import duckdb
import pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"  # mid-panel month -- plenty of client history behind it, not the sealed final month

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    # point straight at the one partition we need -- this is the whole "iterate on a slice" trick
    "fact_march":  f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f"SELECT COUNT(*) FROM {src}").fetchone()[0]
    print(f"{name:12} {n:>12,} rows")


Paste your Hugging Face READ token (hf_...): ··········
dim_clients           104 rows
dim_content       519,606 rows
fact_march      9,841,378 rows


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

1. **One row** represents **one content page for one client on one day**. It comes from `fact_content_daily_performance` (`report_date × client_hash_id × content_hash_id`).

2. **Tables used:** I mainly use `fact_content_daily_performance` (filtered to `month=2026-03`). I join it with `dim_content` for page information and `dim_clients` to check client history. I do **not** use `fact_content_query_90d` because it overlaps with my prediction period and could cause data leakage.

3. **Time window:** I use **March 2026** (`2026-03-01` to `2026-03-31`). I split the month into:

   * **Days 1–15:** the data available when making the prediction.
   * **Days 16–31:** the future results used to evaluate the prediction.


In [8]:
print("Grain = report_date x client_hash_id x content_hash_id, month = 2026-03")


Grain = report_date x client_hash_id x content_hash_id, month = 2026-03


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**4. What I predict:** Since there isn't a real "refresh this page" column, I create a proxy called `is_declining`. A page is marked `1` if its impressions drop by **20% or more** from the first half of March to the second half. These pages are ranked higher for review.

**5. What I exclude:** I do **not** use `fact_content_query_90d` because its 90-day data overlaps with my prediction period, which could cause data leakage.

| Field                                                           | Bucket   | Why                                                             |
| --------------------------------------------------------------- | -------- | --------------------------------------------------------------- |
| `content_hash_id`, `client_hash_id`                             | Context  | IDs used for joins and grouping, not as features                |
| `report_date`                                                   | Context  | Used to split the month into the first and second half          |
| `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` (Days 1–15) | Feature  | Available before making the prediction                          |
| `ga4_data_available`                                            | Feature  | Already known before the prediction                             |
| `gsc_impressions` (Days 16–31)                                  | Label    | Used only to create `is_declining`                              |
| `fact_content_query_90d` columns                                | Excluded | Can cause data leakage because of the overlapping 90-day window |



In [9]:
feature_cols = ["imp_h1", "clicks_h1", "avg_position_h1", "ctr_h1", "ga4_available"]
label_col = "is_declining"
excluded = ["fact_content_query_90d.* (overlapping 90-day window)"]

print("Features:", feature_cols)
print("Label/proxy:", label_col)
print("Excluded:", excluded)

Features: ['imp_h1', 'clicks_h1', 'avg_position_h1', 'ctr_h1', 'ga4_available']
Label/proxy: is_declining
Excluded: ['fact_content_query_90d.* (overlapping 90-day window)']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Query 1 -- grain:** one row really is one page-day. Zero rows back means the grain holds.

In [10]:
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {TABLES['fact_march']}
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()

print(f"Duplicate (date, client, content) combos found: {len(grain_check)}")
grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (date, client, content) combos found: 0


,report_date,client_hash_id,content_hash_id,c


**Query 2 -- row count and date span:** my slice's size and the dates it actually covers.

In [11]:
span = con.sql(f"""
    SELECT COUNT(*)                    AS n_rows,
           MIN(report_date)            AS min_date,
           MAX(report_date)            AS max_date,
           COUNT(DISTINCT client_hash_id)  AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content
    FROM {TABLES['fact_march']}
""").df()

span


,n_rows,min_date,max_date,n_clients,n_content
0,9841378,2026-03-01,2026-03-31,55,331437


**Query 3 -- availability:** filter with `IS TRUE` and see how many rows actually have usable
GA4 data (not just zero-filled placeholders) inside the month.

In [12]:
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
        ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_available
    FROM {TABLES['fact_march']}
""").df()

availability


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,pct_available
0,9841378,413966.0,4.2


### Five features (max) -- the small feature frame

Decision moment: **mid-March (end of day 15)**. Every feature below is built only from
`report_date <= 2026-03-15`, so nothing in it can see the second half of the month I score
pages against.

In [13]:
feats = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_h1,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks      ELSE 0 END) AS clicks_h1,
        AVG(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_avg_position END)       AS avg_position_h1,
        MAX(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)                     AS ga4_available,
        SUM(CASE WHEN report_date >  DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_h2
    FROM {TABLES['fact_march']}
    GROUP BY 1, 2
    HAVING imp_h1 >= 20   -- drop near-zero-traffic pages, too noisy to score either way
""").df()

feats["ctr_h1"] = (feats["clicks_h1"] / feats["imp_h1"]).replace([float("inf")], 0).fillna(0)

print(f"{len(feats):,} pages with enough first-half traffic to score")
feats.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

109,592 pages with enough first-half traffic to score


,client_hash_id,content_hash_id,imp_h1,clicks_h1,avg_position_h1,ga4_available,imp_h2,ctr_h1
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,4173.0,6.0,6.327311,1,2350.0,0.001438
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,245.0,0.0,3.906852,0,208.0,0.000000
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,3705.0,3.0,6.473735,1,1925.0,0.000810
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,2440.0,8.0,7.259861,1,2504.0,0.003279
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,240.0,1.0,3.860842,1,189.0,0.004167


**Five features, one "knowable at decision moment because..." line each:**

1. **`imp_h1`** (impressions, days 1-15) -- knowable because it's already-observed traffic from
   before my mid-month decision point; nothing about it depends on the rest of the month.
2. **`clicks_h1`** (clicks, days 1-15) -- same: fully realized history at the decision moment.
3. **`avg_position_h1`** (average SERP position, days 1-15) -- an already-measured average over
   the same closed window; no second-half data touches it.
4. **`ctr_h1`** (`clicks_h1 / imp_h1`) -- a pure arithmetic transform of two H1-only columns, so
   it inherits their "already observed" status.
5. **`ga4_available`** (GA4 instrumentation flag) -- a fact about whether analytics tracking was
   even switched on for that client; it's fixed metadata, true regardless of what happens later
   in March, so it's knowable before day 1.

## The trap -- deliberate leakage, then removed

Label: did this page's second-half impressions drop 20%+ versus the first half?
`is_declining = imp_h2 < 0.8 * imp_h1`.

First: the **honest** score, using only the five H1/metadata features above.

In [14]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

model_data = feats.copy()
model_data["is_declining"] = (model_data["imp_h2"] < 0.8 * model_data["imp_h1"]).astype(int)

feature_cols = ["imp_h1", "clicks_h1", "avg_position_h1", "ctr_h1", "ga4_available"]
model_data = model_data.dropna(subset=feature_cols)

X = model_data[feature_cols]
y = model_data["is_declining"]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

honest_model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, honest_model.predict_proba(X_te)[:, 1])
print(f"Honest AUC (5 features only): {honest_auc:.3f}")


Honest AUC (5 features only): 0.598


Now the deliberate leak: add `imp_h2` itself (the exact number the label is computed from)
as a "feature." Watch the score jump toward a perfect 1.0 -- because the model isn't learning a
pattern, it's just re-deriving the label from a column that *is* the label's source.

In [15]:
leaky_cols = feature_cols + ["imp_h2"]  # <-- the leak: this IS what the label is built from

X_leak = model_data[leaky_cols]
X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(X_leak, y, test_size=0.25, random_state=42, stratify=y)

leaky_model = LogisticRegression(max_iter=1000).fit(X_tr_l, y_tr_l)
leaky_auc = roc_auc_score(y_te_l, leaky_model.predict_proba(X_te_l)[:, 1])
print(f"Leaky AUC (5 features + imp_h2): {leaky_auc:.3f}")
print(f"Jump: {honest_auc:.3f} -> {leaky_auc:.3f}")


Leaky AUC (5 features + imp_h2): 1.000
Jump: 0.598 -> 1.000


Delete the leak, keep the honest number. **`imp_h2` is never a feature** -- it lives only in
the label-construction step above, exactly like `trend_direction`/`trend_pct` in notebook 02.

In [16]:
# The leak column never appears in the real feature frame -- this is the one we keep and report.
final_feature_frame = model_data[["client_hash_id", "content_hash_id"] + feature_cols + ["is_declining"]]
print(f"Final honest feature frame: {final_feature_frame.shape}")
print(f"Reported score: AUC = {honest_auc:.3f} (NOT the {leaky_auc:.3f} from the leaky version)")
final_feature_frame.head()


Final honest feature frame: (109592, 8)
Reported score: AUC = 0.598 (NOT the 1.000 from the leaky version)


,client_hash_id,content_hash_id,imp_h1,clicks_h1,avg_position_h1,ctr_h1,ga4_available,is_declining
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,4173.0,6.0,6.327311,0.001438,1,1
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,245.0,0.0,3.906852,0.000000,0,0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,3705.0,3.0,6.473735,0.000810,1,1
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,2440.0,8.0,7.259861,0.003279,1,0
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,240.0,1.0,3.860842,0.004167,1,1


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Limitation:** This data only shows **that** a page's impressions dropped, not **why** they dropped or whether refreshing the page would help. The `is_declining` label is only a proxy based on traffic changes. Low-traffic pages may also show drops because of normal variation, even with the `imp_h1 >= 20` filter. In addition, some clients don't have enough data before March, so their pages are not included in the analysis.

All results are based on **observed data** and are meant to **support decisions**, not predict what Google's algorithm will do.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.